# Melanoma Classification — Final Evaluation
Loads the best saved checkpoints from training and produces all final results.

**This notebook never trains — it only evaluates.**
The test set is touched here for the first and only time.

Notebook flow:
1. Setup & reload data
2. Load best checkpoints
3. Confusion matrices
4. ROC curve comparison
5. Threshold sweep
6. Final metrics table
7. Single image prediction
8. Summary report


## 1 · Imports & setup

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import random
from collections import Counter
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from torchvision.transforms import InterpolationMode
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score, roc_curve,
)
from PIL import Image
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = (
    torch.device("mps")  if torch.backends.mps.is_available()  else
    torch.device("cuda") if torch.cuda.is_available()          else
    torch.device("cpu")
)
print(f"Device : {device}")
print(f"Checkpoints expected:")
print(f"  models/simple_cnn_best.pth  — exists: {Path('models/simple_cnn_best.pth').exists()}")
print(f"  models/resnet18_best.pth    — exists: {Path('models/resnet18_best.pth').exists()}")


Device : mps
Checkpoints expected:
  models/simple_cnn_best.pth  — exists: True
  models/resnet18_best.pth    — exists: True


## 2 · Reload test data

In [2]:
IMG_SIZE      = 224
BATCH_SIZE    = 32
NUM_WORKERS   = 4
PIN_MEMORY    = torch.cuda.is_available()
VAL_FRACTION  = 0.5
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

BASE_DIR  = Path().resolve()
DATA_DIR  = BASE_DIR
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR  = DATA_DIR / "test"

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 20, IMG_SIZE + 20),
                       interpolation=InterpolationMode.BILINEAR),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def stratified_split(dataset, val_fraction, seed):
    rng = random.Random(seed)
    class_indices = {}
    for idx, (_, label) in enumerate(dataset.samples):
        class_indices.setdefault(label, []).append(idx)
    val_idx, test_idx = [], []
    for label, indices in class_indices.items():
        indices = indices.copy()
        rng.shuffle(indices)
        split = int(len(indices) * val_fraction)
        val_idx.extend(indices[:split])
        test_idx.extend(indices[split:])
    return val_idx, test_idx

# Load — same seed and split as training notebook
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=eval_transform)
test_pool     = datasets.ImageFolder(TEST_DIR,  transform=eval_transform)

class_names  = train_dataset.classes
class_to_idx = train_dataset.class_to_idx

val_indices, test_indices = stratified_split(test_pool, VAL_FRACTION, SEED)
test_dataset = Subset(test_pool, test_indices)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
)

print(f"Class mapping : {class_to_idx}")
print(f"Test set      : {len(test_dataset)} images  (never seen during training)")


Class mapping : {'benign': 0, 'malignant': 1}
Test set      : 500 images  (never seen during training)


## 3 · Model definitions & load checkpoints

In [3]:
# ── SimpleCNN ─────────────────────────────────────────────────────────────────
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

# ── ResNet18 ──────────────────────────────────────────────────────────────────
def build_resnet18(num_classes=2):
    model = models.resnet18(weights=None)   # no download — we load our own weights
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, num_classes),
    )
    return model

# ── Load checkpoints ──────────────────────────────────────────────────────────
def load_checkpoint(model, path, device):
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state"])
    model.to(device).eval()
    print(f"  Loaded {path}")
    print(f"    saved at epoch  : {checkpoint.get('epoch', '?')}")
    print(f"    best val recall : {checkpoint.get('val_recall', '?'):.4f}")
    print(f"    best val AUC    : {checkpoint.get('val_auc', '?'):.4f}")
    return model

print("Loading checkpoints...")
cnn_model    = load_checkpoint(SimpleCNN(num_classes=2),    "models/simple_cnn_best.pth", device)
resnet_model = load_checkpoint(build_resnet18(num_classes=2), "models/resnet18_best.pth", device)
print("\nBoth models loaded. Ready to evaluate.")


Loading checkpoints...
  Loaded models/simple_cnn_best.pth
    saved at epoch  : 3
    best val recall : 0.8000
    best val AUC    : 0.9445


/var/folders/x2/j0llhrdj7jdcb8yckrnm24vm0000gn/T/ipykernel_4467/3718111140.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location=de

  Loaded models/resnet18_best.pth
    saved at epoch  : 3
    best val recall : 0.9160
    best val AUC    : 0.9711

Both models loaded. Ready to evaluate.


## 4 · Collect test set predictions

In [4]:
def collect_predictions(model, loader, device):
    """Run model on loader, return true labels and malignant probabilities."""
    model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs   = torch.softmax(outputs, dim=1)[:, 1]  # malignant prob
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    return all_labels, all_probs

def get_preds(probs, threshold=0.5):
    return [1 if p >= threshold else 0 for p in probs]

print("Running inference on test set...")
cnn_labels,    cnn_probs    = collect_predictions(cnn_model,    test_loader, device)
resnet_labels, resnet_probs = collect_predictions(resnet_model, test_loader, device)
print(f"Done. {len(cnn_labels)} test images evaluated.")


Running inference on test set...
Done. 500 test images evaluated.


## 5 · Confusion matrices
Rows = actual label, Columns = predicted label.

| | Predicted Benign | Predicted Malignant |
|---|---|---|
| **Actual Benign** | TN ✓ | FP (false alarm) |
| **Actual Malignant** | FN ✗ missed cancer | TP ✓ |

**FN (false negatives) = malignant predicted as benign — the most dangerous error.**
A good melanoma classifier minimises FN even at the cost of more FP.


In [5]:
THRESHOLD = 0.5

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f"Confusion Matrices — Test Set (threshold = {THRESHOLD})", fontsize=13)

results = [
    (axes[0], cnn_labels,    cnn_probs,    "Simple CNN"),
    (axes[1], resnet_labels, resnet_probs, "ResNet18"),
]

for ax, labels, probs, title in results:
    preds = get_preds(probs, THRESHOLD)
    cm    = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, annot_kws={"size": 16, "weight": "bold"})
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("Actual",    fontsize=11)
    ax.set_title(f"{title}\nTP={tp}  TN={tn}  FP={fp}  FN={fn}", fontsize=11)

Path("reports").mkdir(exist_ok=True)
plt.tight_layout()
plt.savefig("reports/confusion_matrices.png", dpi=150)
plt.show()
print("Saved → reports/confusion_matrices.png")


Saved → reports/confusion_matrices.png


/var/folders/x2/j0llhrdj7jdcb8yckrnm24vm0000gn/T/ipykernel_4467/3490420113.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6 · ROC curves
The ROC curve shows the tradeoff between sensitivity (recall) and specificity
at every possible threshold. A model closer to the top-left corner is better.
AUC = area under the curve — 1.0 is perfect, 0.5 is random.


In [6]:
fig, ax = plt.subplots(figsize=(7, 6))

for labels, probs, name, color in [
    (cnn_labels,    cnn_probs,    "Simple CNN", "steelblue"),
    (resnet_labels, resnet_probs, "ResNet18",   "tomato"),
]:
    fpr, tpr, _ = roc_curve(labels, probs)
    auc = roc_auc_score(labels, probs)
    ax.plot(fpr, tpr, color=color, linewidth=2.5,
            label=f"{name}  (AUC = {auc:.4f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random classifier (AUC = 0.50)")
ax.fill_between([0,1], [0,1], alpha=0.05, color="gray")
ax.set_xlabel("False Positive Rate  (1 - Specificity)", fontsize=11)
ax.set_ylabel("True Positive Rate  (Sensitivity / Recall)", fontsize=11)
ax.set_title("ROC Curve — Simple CNN vs ResNet18", fontsize=13)
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("reports/roc_comparison.png", dpi=150)
plt.show()
print("Saved → reports/roc_comparison.png")


Saved → reports/roc_comparison.png


/var/folders/x2/j0llhrdj7jdcb8yckrnm24vm0000gn/T/ipykernel_4467/3453067667.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7 · Threshold sweep
Default threshold of 0.5 is arbitrary. Lowering it increases recall (catches
more malignant cases) at the cost of more false alarms. This table shows the
tradeoff — for a cancer screener, **threshold 0.3–0.4 is often preferred**.


In [7]:
def threshold_row(labels, probs, t):
    preds = get_preds(probs, t)
    cm    = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "Threshold": f"{t:.2f}",
        "Accuracy":  f"{accuracy_score(labels, preds):.4f}",
        "Precision": f"{precision_score(labels, preds, pos_label=1, zero_division=0):.4f}",
        "Recall":    f"{recall_score(labels, preds, pos_label=1, zero_division=0):.4f}",
        "F1":        f"{f1_score(labels, preds, pos_label=1, zero_division=0):.4f}",
        "FN": fn, "FP": fp,
    }

thresholds = [0.3, 0.35, 0.4, 0.45, 0.5]
header = f"{'Threshold':>10} {'Accuracy':>10} {'Precision':>10} {'Recall':>8} {'F1':>8} {'FN':>5} {'FP':>5}"

print("── Simple CNN ───────────────────────────────────────────────────")
print(header); print("-" * 62)
for t in thresholds:
    r = threshold_row(cnn_labels, cnn_probs, t)
    print(f"{r['Threshold']:>10} {r['Accuracy']:>10} {r['Precision']:>10} "
          f"{r['Recall']:>8} {r['F1']:>8} {r['FN']:>5} {r['FP']:>5}")

print("\n── ResNet18 ─────────────────────────────────────────────────────")
print(header); print("-" * 62)
for t in thresholds:
    r = threshold_row(resnet_labels, resnet_probs, t)
    print(f"{r['Threshold']:>10} {r['Accuracy']:>10} {r['Precision']:>10} "
          f"{r['Recall']:>8} {r['F1']:>8} {r['FN']:>5} {r['FP']:>5}")

print("\nFN = malignant images predicted as benign (missed cancer — minimise this)")
print("FP = benign images predicted as malignant (false alarm — less critical)")


── Simple CNN ───────────────────────────────────────────────────
 Threshold   Accuracy  Precision   Recall       F1    FN    FP
--------------------------------------------------------------
      0.30     0.8240     0.7613   0.9440   0.8429    14    74
      0.35     0.8480     0.8042   0.9200   0.8582    20    56
      0.40     0.8520     0.8333   0.8800   0.8560    30    44
      0.45     0.8400     0.8427   0.8360   0.8394    41    39
      0.50     0.8420     0.8670   0.8080   0.8364    48    31

── ResNet18 ─────────────────────────────────────────────────────
 Threshold   Accuracy  Precision   Recall       F1    FN    FP
--------------------------------------------------------------
      0.30     0.8880     0.8593   0.9280   0.8923    18    38
      0.35     0.8900     0.8652   0.9240   0.8936    19    36
      0.40     0.8960     0.8837   0.9120   0.8976    22    30
      0.45     0.9040     0.9040   0.9040   0.9040    24    24
      0.50     0.9060     0.9076   0.9040   0.90

## 8 · Final metrics table

In [8]:
def full_metrics(labels, probs, threshold=0.5):
    preds = get_preds(probs, threshold)
    cm    = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "Accuracy":  accuracy_score(labels, preds),
        "Precision": precision_score(labels, preds, pos_label=1, zero_division=0),
        "Recall":    recall_score(labels, preds, pos_label=1, zero_division=0),
        "F1":        f1_score(labels, preds, pos_label=1, zero_division=0),
        "AUC":       roc_auc_score(labels, probs),
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
    }

cnn_m    = full_metrics(cnn_labels,    cnn_probs)
resnet_m = full_metrics(resnet_labels, resnet_probs)

print("=" * 65)
print("FINAL TEST SET RESULTS  (threshold = 0.5)")
print("=" * 65)
print(f"{'Metric':<12} {'Simple CNN':>14} {'ResNet18':>14} {'Winner':>12}")
print("-" * 65)
for metric in ["Accuracy", "Precision", "Recall", "F1", "AUC"]:
    cv, rv = cnn_m[metric], resnet_m[metric]
    winner = "ResNet18 ✓" if rv > cv else "SimpleCNN ✓" if cv > rv else "tie"
    print(f"{metric:<12} {cv:>14.4f} {rv:>14.4f} {winner:>12}")
print("-" * 65)
for metric in ["TP", "TN", "FP", "FN"]:
    cv, rv = cnn_m[metric], resnet_m[metric]
    better = "← fewer" if metric == "FN" and rv < cv else              "← fewer" if metric == "FN" and cv < rv else ""
    print(f"{metric:<12} {cv:>14} {rv:>14}  {better}")

print("\nDetailed classification reports:")
print("\n── Simple CNN ───────────────────────────────────────────────")
print(classification_report(cnn_labels, get_preds(cnn_probs),
                             target_names=class_names, zero_division=0))
print("── ResNet18 ─────────────────────────────────────────────────")
print(classification_report(resnet_labels, get_preds(resnet_probs),
                             target_names=class_names, zero_division=0))


FINAL TEST SET RESULTS  (threshold = 0.5)
Metric           Simple CNN       ResNet18       Winner
-----------------------------------------------------------------
Accuracy             0.8420         0.9060   ResNet18 ✓
Precision            0.8670         0.9076   ResNet18 ✓
Recall               0.8080         0.9040   ResNet18 ✓
F1                   0.8364         0.9058   ResNet18 ✓
AUC                  0.9385         0.9688   ResNet18 ✓
-----------------------------------------------------------------
TP                      202            226  
TN                      219            227  
FP                       31             23  
FN                       48             24  ← fewer

Detailed classification reports:

── Simple CNN ───────────────────────────────────────────────
              precision    recall  f1-score   support

      benign       0.82      0.88      0.85       250
   malignant       0.87      0.81      0.84       250

    accuracy                           0.8

## 9 · Visual metrics comparison

In [9]:
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1", "AUC"]
cnn_vals    = [cnn_m[m]    for m in metrics_to_plot]
resnet_vals = [resnet_m[m] for m in metrics_to_plot]

x, w = np.arange(len(metrics_to_plot)), 0.35
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, cnn_vals,    w, label="Simple CNN", color="steelblue")
b2 = ax.bar(x + w/2, resnet_vals, w, label="ResNet18",   color="tomato")

ax.set_xticks(x); ax.set_xticklabels(metrics_to_plot, fontsize=11)
ax.set_ylim(0, 1.15); ax.set_ylabel("Score"); ax.grid(axis="y", alpha=0.3)
ax.set_title("Model Comparison — Test Set Metrics", fontsize=13)
ax.legend(fontsize=11)

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("reports/metrics_comparison.png", dpi=150)
plt.show()
print("Saved → reports/metrics_comparison.png")


Saved → reports/metrics_comparison.png


/var/folders/x2/j0llhrdj7jdcb8yckrnm24vm0000gn/T/ipykernel_4467/2666596479.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10 · Single image prediction
Feed any image from the dataset and get a diagnosis with confidence scores
from both models. Change `image_path` to any `.jpg` in your dataset.


In [10]:
def predict_image(image_path, model, transform, device, class_names):
    """Run a single image through the model and return class + probabilities."""
    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(device)  # add batch dim

    model.eval()
    with torch.no_grad():
        outputs = model(tensor)
        probs   = torch.softmax(outputs, dim=1)[0].cpu().numpy()

    pred_idx   = int(np.argmax(probs))
    pred_class = class_names[pred_idx]
    confidence = probs[pred_idx]
    return pred_class, probs, image

# ── Pick a sample image automatically ────────────────────────────────────────
sample_benign    = next((TEST_DIR / "benign").glob("*.jpg"))
sample_malignant = next((TEST_DIR / "malignant").glob("*.jpg"))

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle("Single Image Predictions", fontsize=13)

for row, (img_path, true_label) in enumerate([
    (sample_benign,    "benign"),
    (sample_malignant, "malignant"),
]):
    for col, (model, model_name) in enumerate([
        (cnn_model,    "Simple CNN"),
        (resnet_model, "ResNet18"),
    ]):
        pred_class, probs, image = predict_image(
            img_path, model, eval_transform, device, class_names
        )
        correct = pred_class == true_label
        color   = "green" if correct else "red"

        ax = axes[row][col]
        ax.imshow(image)
        ax.axis("off")
        ax.set_title(
            f"{model_name}\n"
            f"True: {true_label}\n"
            f"Pred: {pred_class}  ({'✓' if correct else '✗'})\n"
            f"Benign: {probs[0]:.3f}  Malignant: {probs[1]:.3f}",
            fontsize=10, color=color
        )

plt.tight_layout()
plt.savefig("reports/single_image_predictions.png", dpi=150)
plt.show()
print("Saved → reports/single_image_predictions.png")


Saved → reports/single_image_predictions.png


/var/folders/x2/j0llhrdj7jdcb8yckrnm24vm0000gn/T/ipykernel_4467/1741567080.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11 · Summary report

In [11]:
print("=" * 65)
print("MELANOMA CLASSIFICATION — FINAL SUMMARY REPORT")
print("=" * 65)

print(f"""
DATASET
  Train : {len(train_dataset):,} images  (benign + malignant)
  Val   : {len(val_indices):,} images  (stratified split from test/)
  Test  : {len(test_dataset):,} images  (held-out, never seen during training)
  Balance: roughly balanced (~50/50) → class weights applied for safety

MODELS
  Simple CNN  — trained from scratch, 25,954 parameters
  ResNet18    — pretrained on ImageNet, 11.2M parameters
                two-stage: frozen backbone (5 ep) → fine-tune layer4 (10 ep)

RESULTS  (threshold = 0.5)
""")

for name, m in [("Simple CNN", cnn_m), ("ResNet18", resnet_m)]:
    print(f"  {name}")
    print(f"    Accuracy  : {m['Accuracy']:.4f}")
    print(f"    Precision : {m['Precision']:.4f}")
    print(f"    Recall    : {m['Recall']:.4f}  ← sensitivity, most critical metric")
    print(f"    F1        : {m['F1']:.4f}")
    print(f"    AUC       : {m['AUC']:.4f}")
    print(f"    FN (missed malignant) : {m['FN']}")
    print()

winner = "ResNet18" if resnet_m["Recall"] > cnn_m["Recall"] else "Simple CNN"
improvement = abs(resnet_m["Recall"] - cnn_m["Recall"])
print(f"CONCLUSION")
print(f"  {winner} achieved higher malignant recall by {improvement:.4f}.")
print(f"  Transfer learning from ImageNet {'improved' if winner == 'ResNet18' else 'did not improve'}")
print(f"  melanoma detection over a from-scratch CNN baseline.")
print()
print("SAVED FILES")
for f in sorted(Path("reports").glob("*.png")):
    print(f"  {f}")
print("=" * 65)


MELANOMA CLASSIFICATION — FINAL SUMMARY REPORT

DATASET
  Train : 9,605 images  (benign + malignant)
  Val   : 500 images  (stratified split from test/)
  Test  : 500 images  (held-out, never seen during training)
  Balance: roughly balanced (~50/50) → class weights applied for safety

MODELS
  Simple CNN  — trained from scratch, 25,954 parameters
  ResNet18    — pretrained on ImageNet, 11.2M parameters
                two-stage: frozen backbone (5 ep) → fine-tune layer4 (10 ep)

RESULTS  (threshold = 0.5)

  Simple CNN
    Accuracy  : 0.8420
    Precision : 0.8670
    Recall    : 0.8080  ← sensitivity, most critical metric
    F1        : 0.8364
    AUC       : 0.9385
    FN (missed malignant) : 48

  ResNet18
    Accuracy  : 0.9060
    Precision : 0.9076
    Recall    : 0.9040  ← sensitivity, most critical metric
    F1        : 0.9058
    AUC       : 0.9688
    FN (missed malignant) : 24

CONCLUSION
  ResNet18 achieved higher malignant recall by 0.0960.
  Transfer learning from Imag